<a href="https://colab.research.google.com/github/jamnalpriyanka21-collab/Careflow/blob/main/Copy_of_01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:

!pip install -q pm4py

In [7]:
from google.colab import files
import pandas as pd
import pm4py

uploaded = files.upload()          # .xes.gz file select karo
path = list(uploaded.keys())[0]
log = pm4py.read_xes(path)
df = log if isinstance(log, pd.DataFrame) else pm4py.convert_to_dataframe(log)

print("Shape (rows, columns):", df.shape)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Saving Sepsis Cases - Event Log (2).xes.gz to Sepsis Cases - Event Log (2).xes (2).gz


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Shape (rows, columns): (15214, 32)


In [8]:
CASE, ACT, TS = "case:concept:name", "concept:name", "time:timestamp"
raw = df.copy()   # raw ko kabhi modify nahi karenge

print("Rows:", len(raw), "| Cases:", raw[CASE].nunique(), "| Activities:", raw[ACT].nunique())

print("\n1) Missing in key columns:")
print(raw[[CASE, ACT, TS]].isna().sum())

print("\n2) Timestamp dtype:", raw[TS].dtype)
print("   Min:", raw[TS].min(), "| Max:", raw[TS].max())

if "lifecycle:transition" in raw.columns:
    print("\n3) Lifecycle values:")
    print(raw["lifecycle:transition"].value_counts(dropna=False))

in_order = raw.groupby(CASE)[TS].apply(lambda s: s.is_monotonic_increasing)
print("\n4) Cases NOT in time order in file:", (~in_order).sum(), "of", len(in_order))

print("5) Events sharing same case+timestamp:",
      raw.duplicated(subset=[CASE, TS], keep=False).sum())
print("6) Events with same case+activity+timestamp:",
      raw.duplicated(subset=[CASE, ACT, TS], keep=False).sum())
print("   Exact duplicate rows:", raw.duplicated().sum())

epc = raw.groupby(CASE).size()
print("\n7) Events per case:")
print(epc.describe())

srt = raw.sort_values([CASE, TS], kind="stable")
print("\n8) First activity per case:")
print(srt.groupby(CASE)[ACT].first().value_counts())
print("\n   Last activity per case:")
print(srt.groupby(CASE)[ACT].last().value_counts())

Rows: 15214 | Cases: 1050 | Activities: 16

1) Missing in key columns:
case:concept:name    0
concept:name         0
time:timestamp       0
dtype: int64

2) Timestamp dtype: datetime64[ns, UTC]
   Min: 2013-11-07 07:18:29+00:00 | Max: 2015-06-05 10:25:11+00:00

3) Lifecycle values:
lifecycle:transition
complete    15214
Name: count, dtype: int64

4) Cases NOT in time order in file: 0 of 1050
5) Events sharing same case+timestamp: 7526
6) Events with same case+activity+timestamp: 14
   Exact duplicate rows: 1

7) Events per case:
count    1050.000000
mean       14.489524
std        11.475941
min         3.000000
25%         9.000000
50%        13.000000
75%        16.000000
max       185.000000
dtype: float64

8) First activity per case:
concept:name
ER Registration     995
Leucocytes           18
IV Liquid            14
CRP                  10
ER Sepsis Triage      7
ER Triage             6
Name: count, dtype: int64

   Last activity per case:
concept:name
Release A           393
Retur

In [9]:
print("A) Exact duplicate rows:")
print(raw[raw.duplicated(keep=False)][[CASE, ACT, TS]])

print("\nB) Same case+activity+timestamp:")
print(raw[raw.duplicated(subset=[CASE, ACT, TS], keep=False)]
      .sort_values([CASE, TS])[[CASE, ACT, TS]])

print("\nC) Activities involved in timestamp ties:")
ties = raw[raw.duplicated(subset=[CASE, TS], keep=False)]
print(ties[ACT].value_counts())

print("\nD) Sample case not starting with ER Registration:")
first = srt.groupby(CASE)[ACT].first()
odd = first[first != "ER Registration"].index
print(srt[srt[CASE] == odd[0]][[CASE, ACT, TS]].head(10))

print("\nE) Longest case:", epc.idxmax(), "->", epc.max(), "events")
print(srt[srt[CASE] == epc.idxmax()][ACT].value_counts())

A) Exact duplicate rows:
     case:concept:name concept:name            time:timestamp
7160                ES   Leucocytes 2014-11-08 04:50:00+00:00
7161                ES   Leucocytes 2014-11-08 04:50:00+00:00

B) Same case+activity+timestamp:
      case:concept:name concept:name            time:timestamp
7160                 ES   Leucocytes 2014-11-08 04:50:00+00:00
7161                 ES   Leucocytes 2014-11-08 04:50:00+00:00
12491               NGA   LacticAcid 2014-06-18 05:00:00+00:00
12494               NGA   LacticAcid 2014-06-18 05:00:00+00:00
5167                 UM   Leucocytes 2014-09-25 09:09:00+00:00
5169                 UM   Leucocytes 2014-09-25 09:09:00+00:00
6058                 YO   Leucocytes 2014-06-05 06:00:00+00:00
6059                 YO          CRP 2014-06-05 06:00:00+00:00
6060                 YO          CRP 2014-06-05 06:00:00+00:00
6061                 YO   Leucocytes 2014-06-05 06:00:00+00:00
5250                 ZM          CRP 2014-04-15 07:00:00+00:00

In [11]:
steps = []
def record(name, d):
    steps.append({"step": name, "rows": len(d), "cases": d[CASE].nunique()})

clean = raw.copy()
clean["_row_order"] = range(len(clean))   # original file order
record("0 raw loaded", clean)

# Rule 1: sort by case, time, then original order (ties ka order file se)
clean = clean.sort_values([CASE, TS, "_row_order"]).reset_index(drop=True)
record("1 sorted (ties keep file order)", clean)

# Rule 2: sirf exact duplicate row hatao (Step 1 mein check karne ke baad)
data_cols = [c for c in clean.columns if c != "_row_order"]
n_dup = int(clean.duplicated(subset=data_cols).sum())
clean = clean.drop_duplicates(subset=data_cols, keep="first")
record(f"2 dropped {n_dup} exact duplicate row(s)", clean)

# Rule 3: 55 odd-start cases ko DELETE nahi, FLAG karo
clean["event_idx"] = clean.groupby(CASE).cumcount() + 1
first_act = clean.groupby(CASE)[ACT].transform("first")
clean["starts_with_registration"] = first_act.eq("ER Registration")

log_df = pd.DataFrame(steps)
print(log_df)
clean.to_csv("event_log_clean.csv", index=False)
log_df.to_csv("cleaning_log.csv", index=False)

                                 step   rows  cases
0                        0 raw loaded  15214   1050
1     1 sorted (ties keep file order)  15214   1050
2  2 dropped 1 exact duplicate row(s)  15213   1050


In [12]:
dd = pd.DataFrame({
    "column": clean.columns,
    "dtype": clean.dtypes.astype(str).values,
    "non_null": clean.notna().sum().values,
    "missing_pct": (clean.isna().mean() * 100).round(1).values,
    "n_unique": clean.nunique().values,
})
dd["example_values"] = [", ".join(map(str, clean[c].dropna().unique()[:3]))
                        for c in clean.columns]
dd["meaning"] = ""     # nahi pata to "not documented" likho, andaza mat lagao
dd.to_csv("data_dictionary_draft.csv", index=False)
dd

,column,dtype,non_null,missing_pct,n_unique,example_values,meaning
0,InfectionSuspected,object,1050,93.1,2,"True, False",
1,org:group,object,15213,0.0,26,"A, B, C",
2,DiagnosticBlood,object,1050,93.1,2,"True, False",
3,DisfuncOrg,object,1050,93.1,2,"True, False",
4,SIRSCritTachypnea,object,1050,93.1,2,"True, False",
5,Hypotensie,object,1050,93.1,2,"True, False",
6,SIRSCritHeartRate,object,1050,93.1,2,"True, False",
7,Infusion,object,1050,93.1,2,"True, False",
8,DiagnosticArtAstrup,object,1050,93.1,2,"True, False",
9,concept:name,object,15213,0.0,16,"ER Registration, Leucocytes, CRP",
